In [2]:
from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# define the schema
schema = StructType() \
.add("ProductID", IntegerType(), True) \
.add("ProductName", StringType(), True) \
.add("Category", StringType(), True) \
.add("ListPrice", DoubleType(), True)

df = spark.read.format("csv").option("header","true").schema(schema).load("Files/Data/Products/products.csv")
# df now is a Spark DataFrame containing CSV data from "Files/products/products.csv".
display(df)

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 947ac86c-a81d-4e80-97de-8b8fd18a4c7f)

In [3]:
df.write.format("delta").saveAsTable("managed_products")

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 5, Finished, Available, Finished, False)

In [7]:
df.write.format("delta").saveAsTable("external_products", path="abfss://Testeo@onelake.dfs.fabric.microsoft.com/LH_TEST.Lakehouse/Files/external_products")

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 11, Finished, Available, Finished, False)

IllegalArgumentException: requirement failed: external_products has invalid metadata at metadataLocation: b84c0454-339a-41cf-8dd8-d414e7ac0d0a/Tables/dbo/external_products, it's neither a view nor a table

In [5]:
# Verificar que existe y ver su contenido
spark.sql("SELECT * FROM managed_products LIMIT 10").show()

# O con el DataFrame API
df = spark.table("managed_products")
df.show()

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 8, Finished, Available, Finished, False)

+---------+--------------------+--------------+---------+
|ProductID|         ProductName|      Category|ListPrice|
+---------+--------------------+--------------+---------+
|      771|Mountain-100 Silv...|Mountain Bikes|  3399.99|
|      772|Mountain-100 Silv...|Mountain Bikes|  3399.99|
|      773|Mountain-100 Silv...|Mountain Bikes|  3399.99|
|      774|Mountain-100 Silv...|Mountain Bikes|  3399.99|
|      775|Mountain-100 Blac...|Mountain Bikes|  3374.99|
|      776|Mountain-100 Blac...|Mountain Bikes|  3374.99|
|      777|Mountain-100 Blac...|Mountain Bikes|  3374.99|
|      778|Mountain-100 Blac...|Mountain Bikes|  3374.99|
|      779|Mountain-200 Silv...|Mountain Bikes|  2319.99|
|      780|Mountain-200 Silv...|Mountain Bikes|  2319.99|
+---------+--------------------+--------------+---------+

+---------+--------------------+--------------+---------+
|ProductID|         ProductName|      Category|ListPrice|
+---------+--------------------+--------------+---------+
|      771|Mo

In [8]:
%%sql
DESCRIBE FORMATTED managed_products;

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 12, Finished, Available, Finished, False)

<Spark SQL result set with 11 rows and 3 fields>

In [9]:
%%sql
DROP TABLE managed_products;

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [20]:
%%sql
CREATE TABLE products
USING DELTA
LOCATION 'Files/external_products';

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 24, Finished, Available, Finished, False)

Error: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `chimcobldhq2al35edq6arp59h45ul25ada2ap32ds`.`products` because it already exists.
Choose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects.

In [21]:
%%sql
SELECT * FROM products;

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [22]:
%%sql
UPDATE products
SET ListPrice = ListPrice * 0.9
WHERE Category = 'Mountain Bikes';

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 26, Finished, Available, Finished, False)

Error: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `ListPrice` cannot be resolved. Did you mean one of the following? [`Price`, `Category`, `Productid`, `ProductName`].; line 2 pos 4;
'DeltaUpdateTable ['ListPrice], [('ListPrice * 0.9)], (Category#2932 = Mountain Bikes)
+- SubqueryAlias spark_catalog.chimcobldhq2al35edq6arp59h45ul25ada2ap32ds.products
   +- Relation spark_catalog.chimcobldhq2al35edq6arp59h45ul25ada2ap32ds.products[Productid#2930,ProductName#2931,Category#2932,Price#2933] parquet


In [23]:
%%sql
DESCRIBE HISTORY products;

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 27, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 15 fields>

In [24]:
delta_table_path = 'Files/external_products'
# Get the current data
current_data = spark.read.format("delta").load(delta_table_path)
display(current_data)

# Get the version 0 data
original_data = spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path)
display(original_data)

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 28, Finished, Available, Finished, False)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: Files/external_products.

In [25]:
%%sql
-- Create a temporary view
CREATE OR REPLACE TEMPORARY VIEW products_view
AS
    SELECT Category, COUNT(*) AS NumProducts, MIN(ListPrice) AS MinPrice, MAX(ListPrice) AS MaxPrice, AVG(ListPrice) AS AvgPrice
    FROM products
    GROUP BY Category;

SELECT *
FROM products_view
ORDER BY Category;    

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 29, Finished, , Finished, True)

Error: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `ListPrice` cannot be resolved. Did you mean one of the following? [`Price`, `Category`, `Productid`, `ProductName`].; line 4 pos 50;
'CreateViewCommand `products_view`, SELECT Category, COUNT(*) AS NumProducts, MIN(ListPrice) AS MinPrice, MAX(ListPrice) AS MaxPrice, AVG(ListPrice) AS AvgPrice
    FROM products
    GROUP BY Category, false, true, LocalTempView, false
+- 'Aggregate [Category#3148], [Category#3148, count(1) AS NumProducts#3142L, 'MIN('ListPrice) AS MinPrice#3143, 'MAX('ListPrice) AS MaxPrice#3144, 'AVG('ListPrice) AS AvgPrice#3145]
   +- SubqueryAlias spark_catalog.chimcobldhq2al35edq6arp59h45ul25ada2ap32ds.products
      +- Relation spark_catalog.chimcobldhq2al35edq6arp59h45ul25ada2ap32ds.products[Productid#3146,ProductName#3147,Category#3148,Price#3149] parquet


In [26]:
%%sql
SELECT Category, NumProducts
FROM products_view
ORDER BY NumProducts DESC
LIMIT 10;

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 30, Finished, Available, Finished, False)

Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `products_view` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 2 pos 5;
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Sort ['NumProducts DESC NULLS LAST], true
      +- 'Project ['Category, 'NumProducts]
         +- 'UnresolvedRelation [products_view], [], false


In [27]:
from pyspark.sql.functions import col, desc

df_products = spark.sql("SELECT Category, MinPrice, MaxPrice, AvgPrice FROM products_view").orderBy(col("AvgPrice").desc())
display(df_products.limit(6))

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 31, Finished, Available, Finished, False)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `products_view` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 51;
'Project ['Category, 'MinPrice, 'MaxPrice, 'AvgPrice]
+- 'UnresolvedRelation [products_view], [], false


In [28]:
from notebookutils import mssparkutils
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Create a folder
inputPath = 'Files/data/'
mssparkutils.fs.mkdirs(inputPath)

# Create a stream that reads data from the folder, using a JSON schema
jsonSchema = StructType([
StructField("device", StringType(), False),
StructField("status", StringType(), False)
])
iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 1).json(inputPath)

# Write some event data to the folder
device_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "data.txt", device_data, True)

print("Source stream created...")

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 32, Finished, Available, Finished, False)

Source stream created...


In [29]:
# Write the stream to a delta table
delta_stream_table_path = 'Tables/iotdevicedata'
checkpointpath = 'Files/delta/checkpoint'
deltastream = iotstream.writeStream.format("delta").option("checkpointLocation", checkpointpath).start(delta_stream_table_path)
print("Streaming to delta sink...")

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 33, Finished, Available, Finished, False)

Streaming to delta sink...


In [30]:
%%sql
SELECT * FROM IotDeviceData;

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 34, Finished, Available, Finished, False)

Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `IotDeviceData` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [IotDeviceData], [], false


In [31]:
# Add more data to the source stream
more_data = '''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "more-data.txt", more_data, True)

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 35, Finished, Available, Finished, False)

True

In [32]:
%%sql
SELECT * FROM IotDeviceData;

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 36, Finished, Available, Finished, False)

Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `IotDeviceData` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [IotDeviceData], [], false


In [33]:
deltastream.stop()

StatementMeta(, aef28dd0-1511-4db7-9696-58753b4e79ba, 37, Finished, Available, Finished, False)